In [4]:
!pip install pymysql
#디비 연결을 위해 필요한 패키지, 한번만 실행

In [1]:
#행렬과 관련된 수학식을 사용할 수 있는 라이브러리
import numpy as np

#데이터분석용 pandas 사용을 위한 라이브러리
import pandas as pd

#데이터베이스 연결 위한 라이브러리
from pymysql import connect, cursors

#그래프, 시각화를 위한 라이브러리
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

#문자데이터를 숫자로 인코딩하기 위한 라이브러리
from sklearn.preprocessing import LabelEncoder

#전체 데이터를 학습용 데이터와 검증용 데이터로 나누기 위한 라이브러리
from sklearn.model_selection import train_test_split

#분석 평가 지표 사용을 위한 라이브러리
from sklearn.metrics import  mean_absolute_error, mean_squared_error, r2_score

In [2]:
fm._load_fontmanager(try_read_cache=False)
plt.rc("font", family="NanumGothic")
plt.rc("axes", unicode_minus=False)

In [3]:
conn = connect(
    host='192.168.30.79',       
    port=3306,
    user='user',
    password='mariadb',
    database='project-2',
    charset='utf8mb4'       # 한글 깨짐 방지
)

cursor = conn.cursor()

cursor.execute("SELECT 1")
print(cursor.fetchone())    # (1,) 이 나오면 연결 성공

(1,)


In [4]:
df = pd.read_csv('가스_인구_기온_통합.csv',  encoding='cp949')
df

,연월,시도,월평균기온,월평균최고기온,월평균최저기온,가구수(개),공급량(㎥),총인구수
0,2021-01,강원,-4.2,1.5,-9.7,386599,51918.194190,1541696
1,2021-02,강원,1.3,7.3,-4.6,387774,43068.962620,1540875
2,2021-03,강원,7.0,13.3,1.3,388403,32782.803000,1536399
3,2021-04,강원,11.8,17.9,5.7,390610,20762.350470,1536175
4,2021-05,강원,15.3,21.1,10.0,391098,14472.110240,1535491
...,...,...,...,...,...,...,...,...
1183,2026-02,충북,1.1,7.7,-5.0,584666,47598.860310,1596816
1184,2026-03,충북,6.8,13.6,0.5,579452,39099.270910,1597976
1185,2026-04,충북,13.5,20.6,6.4,576171,21893.604080,1599375
1186,2026-05,충북,18.6,25.5,11.8,576065,12157.524940,1600216


In [5]:
cursor = conn.cursor()

# region 테이블에서 (번호, 이름) 전부 가져오기
cursor.execute("SELECT region_id, region_name FROM region")
rows = cursor.fetchall()          # ((1, '서울'), (2, '부산'), ...)

# 변환표 만들기: {'서울': 1, '부산': 2, ...}
region_map = {}
for rid, name in rows:
    region_map[name] = rid
print(region_map)

{'강원': 10, '경기': 9, '경남': 16, '경북': 15, '광주': 5, '대구': 3, '대전': 6, '부산': 2, '서울': 1, '세종': 8, '울산': 7, '인천': 4, '전국': 18, '전남': 14, '전북': 13, '제주': 17, '충남': 12, '충북': 11}


In [7]:
# '시도' 컬럼의 이름을 변환표로 번호로 바꿔서 새 컬럼에 저장
df['region_id'] = df['시도'].map(region_map)

# ★ 꼭 확인: 변환표에 없는 이름이 있으면 region_id 가 비어(NaN) 있음
print(df[df['region_id'].isna()]['시도'].unique())  # [] 이면 전부 매칭 성공
df


[]


,연월,시도,월평균기온,월평균최고기온,월평균최저기온,가구수(개),공급량(㎥),총인구수,region_id
0,2021-01,강원,-4.2,1.5,-9.7,386599,51918.194190,1541696,10
1,2021-02,강원,1.3,7.3,-4.6,387774,43068.962620,1540875,10
2,2021-03,강원,7.0,13.3,1.3,388403,32782.803000,1536399,10
3,2021-04,강원,11.8,17.9,5.7,390610,20762.350470,1536175,10
4,2021-05,강원,15.3,21.1,10.0,391098,14472.110240,1535491,10
...,...,...,...,...,...,...,...,...,...
1183,2026-02,충북,1.1,7.7,-5.0,584666,47598.860310,1596816,11
1184,2026-03,충북,6.8,13.6,0.5,579452,39099.270910,1597976,11
1185,2026-04,충북,13.5,20.6,6.4,576171,21893.604080,1599375,11
1186,2026-05,충북,18.6,25.5,11.8,576065,12157.524940,1600216,11


In [8]:
cursor = conn.cursor()
insert_sql = "INSERT INTO GAS_MONTHLY (REGION_ID, YM, AVG_TEMP, MAX_TEMP, MIN_TEMP, HOUSEHOLD_CNT, POPULATION, SUPPLY) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"
insert_param = []
for e in df.values:
  each_data = (e[8], e[0], e[2], e[3], e[4], e[5], e[7], e[6])
  insert_param.append(each_data)

#쿼리 실행
cursor.executemany(insert_sql, insert_param)
#커밋
conn.commit()

In [12]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


models = {}
result_anls = []
train_col = ['월평균기온', '총인구수']

for region in df['시도'].unique():
    sub = df[df['시도'] == region]

    #랜덤포레스트 객체 생성
    rf_model = RandomForestRegressor(
      n_estimators= 100, #트리를 몇 개 만들지 지정
      random_state= 42,
      n_jobs = -1
    )

    #xg_boost 객체 생성
    xg_model = XGBRegressor(
      n_estimators=100,
      learning_rate=0.1,
      random_state=42,
      n_jobs=-1
    )


    #학습데이터와 검증데이터를 분리
    train_df, test_df = train_test_split(sub, test_size=0.2, random_state=42)
    X_train = train_df[train_col]

    #학습에 대한 답 데이터(train_df의 발전량 데이터)
    y_train = train_df['공급량(㎥)']

    #검증용 데이터
    X_test = test_df[train_col]
    y_test = test_df['공급량(㎥)']

    #랜덤포레스트 학습
    rf_model.fit(X_train, y_train)

    #예측
    rf_predict = rf_model.predict(X_test)

    #XGBoost 학습
    xg_model.fit(X_train, y_train)

    #예측
    xg_predict = xg_model.predict(X_test)

    print(f"[{region}] RF 학습완료, XGBoost 학습완료")

    #랜덤포레스트 결과 분석
    rf_mae = mean_absolute_error(y_test, rf_predict)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predict))
    rf_r2 = r2_score(y_test, rf_predict)

    #xg부스터 결과 분석
    xg_mae = mean_absolute_error(y_test, xg_predict)
    xg_rmse = np.sqrt(mean_squared_error(y_test, xg_predict))
    xg_r2 = r2_score(y_test, xg_predict)

    print(f'랜덤포레스트 MAE = {rf_mae}, RMSE = {rf_rmse}, R2 = {rf_r2}')
    print(f'XGBOOST MAE = {xg_mae}, RMSE = {xg_rmse}, R2 = {xg_r2}')

    #모델 저장
    models[region] = rf_model
    result_anls.append({'region': region, 
                         'rf_mae': rf_mae, 'rf_rmse': rf_rmse, 'rf_r2' : rf_r2,
                         'xg_mae': xg_mae, 'xg_rmse': xg_rmse, 'xg_r2' : xg_r2})

    #공급량의 값이 계절에 따라 급락이 큼 오차가 크게 나옴

[강원] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 3847.1077478135676, RMSE = 5041.883427547691, R2 = 0.9128869924072698
XGBOOST MAE = 5661.600893330357, RMSE = 7074.711136727622, R2 = 0.8284799747484094
[경기] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 24852.53772319292, RMSE = 31775.80070327046, R2 = 0.9665844829058131
XGBOOST MAE = 30722.182057232138, RMSE = 42787.801240564884, R2 = 0.9394108137544575
[경남] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 10152.165467392846, RMSE = 12976.849047993517, R2 = 0.875281388342332
XGBOOST MAE = 12774.266371763393, RMSE = 17682.224213275415, R2 = 0.7684383963477758
[경북] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 5076.299049684988, RMSE = 6783.061039551503, R2 = 0.9613946210944715
XGBOOST MAE = 6429.728961964285, RMSE = 8531.82390323902, R2 = 0.9389226571476156
[광주] RF 학습완료, XGBoost 학습완료
랜덤포레스트 MAE = 2616.543571428572, RMSE = 3271.7930339090394, R2 = 0.9796059811031664
XGBOOST MAE = 2624.4683663504466, RMSE = 3223.441005680391, R2 = 0.9802043109550321
[대구] RF 학습완료, XGBoost 학습완료
랜덤포레스트

In [6]:
target = next((e for e in result_anls if e['region'] == '제주'), None)

print(target)
print(result_anls)

{'region': '제주', 'rf_mae': 328.80260345238054, 'rf_rmse': np.float64(526.0570758385576), 'rf_r2': 0.7456948467139048, 'xg_mae': 348.54130998883926, 'xg_rmse': np.float64(553.9241878378683), 'xg_r2': 0.7180383216390569}
[{'region': '강원', 'rf_mae': 3961.8235005126876, 'rf_rmse': np.float64(5415.0557840476995), 'rf_r2': 0.8995145278326103, 'xg_mae': 5179.19694897768, 'xg_rmse': np.float64(6652.8252574026665), 'xg_r2': 0.8483265237633748}, {'region': '경기', 'rf_mae': 19176.707035781263, 'rf_rmse': np.float64(24890.539832388018), 'rf_r2': 0.9794966978499572, 'xg_mae': 31294.27181026786, 'xg_rmse': np.float64(42907.1244639701), 'xg_r2': 0.9390724099291169}, {'region': '경남', 'rf_mae': 7113.056995833937, 'rf_rmse': np.float64(9171.665034473786), 'rf_r2': 0.937699844239952, 'xg_mae': 5833.636006696428, 'xg_rmse': np.float64(7617.077660308823), 'xg_r2': 0.9570295868678165}, {'region': '경북', 'rf_mae': 4978.1351404394945, 'rf_rmse': np.float64(6895.548388096197), 'rf_r2': 0.9601035742573149, 'xg_ma

In [13]:
# '2021-01' → 1 처럼 월 숫자 컬럼 만들기
df['월'] = df['연월'].str[5:].astype(int)

future_months = ['2026-07', '2026-08', '2026-09', '2026-10', '2026-11', '2026-12']


rows = []
for region, model in models.items():
    sub = df[df['시도'] == region]

    # 평년 기온: 이 지역의 월별 평균 기온 (1월 평균, 2월 평균, ...)
    normal_temp = sub.groupby('월')['월평균기온'].mean()
    # 이 지역의 가장 최근 인구수
    last_pop = sub.sort_values('연월')['총인구수'].iloc[-1]

    for ym in future_months:
        month = int(ym[5:])            # '2026-07' → 7
        temp = normal_temp[month]      # 7월 평년 기온

        # 학습할 때와 같은 컬럼 이름으로 입력을 만들어야 함
        X_future = pd.DataFrame({'월평균기온': [temp],  '총인구수': [last_pop]})
        pred = model.predict(X_future)[0]

        rows.append({'시도': region, '연월': ym, '예상기온': temp, '예측공급량': pred})

forecast_df = pd.DataFrame(rows)

In [14]:
# 6개월 예측 (서울)
print(forecast_df[forecast_df['시도'] == '서울'])

# 3개월 예측 = 앞의 3개 달만
first3 = ['2026-07', '2026-08', '2026-09']
print(forecast_df[(forecast_df['시도'] == '서울') & (forecast_df['연월'].isin(first3))])

    시도       연월   예상기온          예측공급량
48  서울  2026-07  27.50   43162.048252
49  서울  2026-08  27.26   43506.528795
50  서울  2026-09  23.54   47635.407027
51  서울  2026-10  15.84  156963.146231
52  서울  2026-11   8.74  231961.690127
53  서울  2026-12   0.18  391384.190951
    시도       연월   예상기온         예측공급량
48  서울  2026-07  27.50  43162.048252
49  서울  2026-08  27.26  43506.528795
50  서울  2026-09  23.54  47635.407027
